# Overton Window — Baseline Exploration

**SI 630: Natural Language Processing · Winter 2026**  
**Paris Heard · School of Information, University of Michigan**

---

This notebook documents the **initial exploratory pipeline** built on the Babel Briefings dataset before the final dataset was identified. It is included for methodological transparency: understanding what was tried, what it revealed, and why it was replaced is part of the research record.

### What It Covers
1. Loading and inspecting the Babel Briefings corpus (54 multilingual JSON files)
2. Filtering to US English headlines
3. Implementing the TF-IDF centroid distance framework as proof-of-concept
4. Evaluating results and identifying dataset limitations
5. Documenting the decision to switch to the US Multi-Outlet News Headlines dataset

### Why Babel Briefings Was Replaced
The Babel Briefings corpus covers only **August 2020 – November 2021** (16 monthly bins), which is far too short to measure the long-term discourse drift the research question requires. The US English subset contains ~92,443 headlines, compared to 5,804,010 in the final dataset. Results on Babel Briefings provided proof-of-concept validation (the November 2021 Omicron spike was cleanly detected), but the temporal range made it unsuitable as a primary corpus.

**The final analysis uses:** `dess-mannheim/US_Multi_Outlet_News_Headlines2001_2024` (see `02_main_analysis.ipynb`)

## 0. Imports & Setup

In [ ]:
import os
import re
import json
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Preprocessing parameters (carried forward to final analysis)
MIN_WORDS = 4

print('Imports OK')

## 1. Data Loading — Babel Briefings

Babel Briefings (Leeb & Schölkopf, 2024) is a multilingual news headline corpus distributed as 54 JSON files organized by country/language code. Each file contains a list of article objects with fields including `title`, `publishedAt`, `language`, and `source-name`.

**Data directory:** `./babel_briefings/` (not included in repo — see `data/README.md`)

In [ ]:
DATA_DIR = './babel_briefings/'

# recursively find all JSON files
json_files = glob.glob(os.path.join(DATA_DIR, '**', '*.json'), recursive=True)
print(f'Found {len(json_files)} JSON files')
print('Example paths:', json_files[:3])

In [ ]:
# inspect schema of first file
with open(json_files[0], 'r') as f:
    sample = json.load(f)

# handle both list-of-articles and dict-wrapped formats
if isinstance(sample, list):
    sample_articles = sample
elif isinstance(sample, dict):
    # common wrapper keys include 'articles', 'data', 'items', etc.
    sample_articles = next(iter(sample.values()))

print(f'Records in first file: {len(sample_articles)}')
print(f'Field names: {list(sample_articles[0].keys())}')
print('\nSample record:')
print(json.dumps({k: sample_articles[0][k] for k in ['title', 'publishedAt', 'language', 'source-name']}, indent=2))

In [ ]:
# Load all files, filter to English headlines
all_records = []
for path in json_files:
    with open(path, 'r') as f:
        data = json.load(f)
    for item in data:
        if item.get('language') == 'en' and item.get('title'):
            all_records.append({
                'title':       item['title'],
                'date':        item.get('publishedAt') or item.get('collectedAt'),
                'source':      item.get('source-name', 'unknown'),
                'language':    item.get('language'),
            })

df_babel = pd.DataFrame(all_records)
print(f'Total English records: {len(df_babel):,}')
print(f'Unique sources: {df_babel["source"].nunique()}')
print(df_babel.head())

## 2. Preprocessing

Same pipeline applied in the final analysis: lowercase, strip URLs, keep letters/digits/hyphens, enforce minimum word count.

In [ ]:
def parse_date_babel(date_str):
    """Parse ISO 8601 dates from Babel Briefings to YYYY-MM."""
    try:
        return pd.to_datetime(date_str).strftime('%Y-%m')
    except:
        return None

def preprocess(text):
    """Normalize headline: lowercase, strip URLs and punctuation, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r'http\S+',        '', text)
    text = re.sub(r'[^a-z0-9\s\-]', '', text)
    text = re.sub(r'\s+',            ' ', text).strip()
    return text

def is_valid(text, min_words=MIN_WORDS):
    """Return True if headline meets minimum word count after preprocessing."""
    return len(str(text).split()) >= min_words

df_babel['year_month']   = df_babel['date'].apply(parse_date_babel)
df_babel['title_clean']  = df_babel['title'].apply(preprocess)
df_babel = df_babel[df_babel['title_clean'].apply(is_valid)].dropna(subset=['year_month'])
df_babel = df_babel.sort_values('year_month').reset_index(drop=True)

print(f'After filtering: {len(df_babel):,} headlines')
print(f'Date range: {df_babel["year_month"].min()} to {df_babel["year_month"].max()}')
print(f'Monthly bins: {df_babel["year_month"].nunique()}')

## 3. TF-IDF Proof-of-Concept

The first 12 months are used as a baseline; the remaining 4 as the comparison window.

> **Note:** This split is purely exploratory — the Babel Briefings date range (Aug 2020 – Nov 2021) does not map onto a theoretically motivated baseline/comparison division. The short window was the primary reason for switching datasets.

In [ ]:
months_sorted = sorted(df_babel['year_month'].unique())
baseline_months   = months_sorted[:12]
comparison_months = months_sorted[12:]

print(f'Baseline months  : {baseline_months[0]} – {baseline_months[-1]}')
print(f'Comparison months: {comparison_months[0]} – {comparison_months[-1]}')

df_base = df_babel[df_babel['year_month'].isin(baseline_months)]
df_comp = df_babel[df_babel['year_month'].isin(comparison_months)]

print(f'\nBaseline   : {len(df_base):,} headlines')
print(f'Comparison : {len(df_comp):,} headlines')

In [ ]:
# Fit vectorizer on baseline only (same design decision as final analysis)
vectorizer = TfidfVectorizer(
    max_features = 5000,
    ngram_range  = (1, 2),
    min_df       = 3,
    sublinear_tf = True,
    stop_words   = 'english',
)
X_base = vectorizer.fit_transform(df_base['title_clean'])
X_comp = vectorizer.transform(df_comp['title_clean'])

centroid_baseline = np.asarray(X_base.mean(axis=0)).flatten()

print(f'Vocabulary size: {len(vectorizer.get_feature_names_out()):,}')
print(f'Baseline matrix: {X_base.shape}')

In [ ]:
# Monthly cosine distance from baseline centroid
records = []
for corpus_label, df, X in [('baseline', df_base, X_base), ('comparison', df_comp, X_comp)]:
    for month, group in df.groupby('year_month'):
        idx  = group.index.tolist()
        mat  = X[idx]
        ctrd = np.asarray(mat.mean(axis=0)).flatten()
        sim  = cosine_similarity(ctrd.reshape(1,-1), centroid_baseline.reshape(1,-1))[0,0]
        dist = 1.0 - float(sim)
        records.append({'month': month, 'cosine_dist': dist, 'corpus': corpus_label, 'n': len(idx)})

df_results = pd.DataFrame(records).sort_values('month').reset_index(drop=True)
print(df_results.to_string(index=False))

## 4. Results & Limitations

The Babel Briefings proof-of-concept confirmed that the centroid distance framework is sensitive to genuine discourse events. The November 2021 Omicron spike (distance = 0.121) was cleanly detected. However, two critical limitations ruled it out as the primary corpus:

1. **Temporal range too short** — 16 monthly bins cannot support a study of long-term discourse drift. The research question requires at minimum a decade of baseline data.
2. **No pre-2020 coverage** — The Overton Window shift hypothesis concerns the post-2016 political period, which requires a stable pre-2016 reference corpus.

These limitations motivated the switch to the US Multi-Outlet News Headlines dataset, which covers 2001–2024 across five major US outlets.

In [ ]:
# Visualize proof-of-concept results
all_months = sorted(df_results['month'].unique())
m2x = {m: i for i, m in enumerate(all_months)}

fig, ax = plt.subplots(figsize=(12, 4))
fig.suptitle('Babel Briefings — Proof-of-Concept: Cosine Distance from Baseline Centroid', fontsize=12)

for corpus, color in [('baseline', 'steelblue'), ('comparison', 'darkorange')]:
    sub = df_results[df_results['corpus'] == corpus]
    x   = [m2x[m] for m in sub['month']]
    ax.plot(x, sub['cosine_dist'], color=color, marker='o', linewidth=1.5, label=corpus.capitalize())

ax.set_xticks(range(len(all_months)))
ax.set_xticklabels(all_months, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Cosine Distance from Baseline Centroid')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

Path('outputs/figures').mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig('outputs/figures/babel_proof_of_concept.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: babel_proof_of_concept.png')

## 5. Dataset Search & Decision Log

The following datasets were evaluated before selecting the final corpus:

| Dataset | Reason Rejected |
|---|---|
| Stanford OVAL CC-News (600M articles, 2016–2024) | No North America filter; no pre-2016 coverage |
| 3DLNews2 (US local news, 1995–2024) | 7GB Globus download; local-only coverage |
| HEADLINES semantic similarity (Dell/Harvard, 1920–1989) | Date range mismatch |
| Factiva RTF exports (via U-M library) | 100-article-per-download limit made bulk acquisition infeasible |
| **US Multi-Outlet News Headlines 2001–2024** | **Selected: temporal range, outlet diversity, US-specificity, HuggingFace streaming** |

**→ Continue to `02_main_analysis.ipynb` for the full pipeline.**